In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target = DeltaTable.forName(
    spark,
    "workspace.silver.customers_scd2"
)

source = (
    spark.table("workspace.silver.customers")
    .withColumn(
        "effective_start_date",
        F.current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        F.to_timestamp(F.lit("9999-12-31 23:59:59"))
    )
    .withColumn("is_current", F.lit(True))
)

# Detect changed customer records
changed = (
    source.alias("s")
    .join(
        spark.table("workspace.silver.customers_scd2")
        .filter("is_current = true")
        .alias("t"),
        "customer_id"
    )
    .filter(
        (F.col("s.customer_segment") != F.col("t.customer_segment")) |
        (F.col("s.city") != F.col("t.city")) |
        (F.col("s.state") != F.col("t.state")) |
        (F.col("s.email") != F.col("t.email"))
    )
    .select("s.*")
)

display(changed)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target = DeltaTable.forName(
    spark,
    "workspace.silver.customers_scd2"
)

source = (
    spark.table("workspace.silver.customers")
    .withColumn("effective_start_date", F.current_timestamp())
    .withColumn(
        "effective_end_date",
        F.to_timestamp(F.lit("9999-12-31 23:59:59"))
    )
    .withColumn("is_current", F.lit(True))
)

# Expire existing versions where attributes changed
target.alias("t").merge(
    source.alias("s"),
    """
    t.customer_id = s.customer_id
    AND t.is_current = true
    """
).whenMatchedUpdate(
    condition="""
    t.customer_segment <> s.customer_segment
    OR t.city <> s.city
    OR t.state <> s.state
    OR t.email <> s.email
    """,
    set={
        "is_current": "false",
        "effective_end_date": "current_timestamp()"
    }
).execute()

# Insert new current versions
current_customers = (
    source.alias("s")
    .join(
        spark.table("workspace.silver.customers_scd2")
        .filter("is_current = true")
        .select("customer_id")
        .alias("t"),
        "customer_id",
        "left_anti"
    )
)

(
    current_customers
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.silver.customers_scd2")
)

In [0]:
%sql
SELECT
    customer_id,
    customer_segment,
    effective_start_date,
    effective_end_date,
    is_current
FROM workspace.silver.customers_scd2
ORDER BY customer_id, effective_start_date;